# PySpark 3: joins and aggregations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-03-joins-and-aggregations.ipynb)

Finance wants revenue by region and status, and a list of customers who have never ordered. Both answers
depend on joins that keep the right rows and aggregates that count each order exactly once.

This module chooses join types by what should happen to unmatched rows, handles null and duplicate keys,
broadcasts small dimensions, and aggregates with `groupBy`, `pivot` and `rollup`. Before every join, ask what
one row of each input represents, its **grain**, and what one row of the output should represent.

Each section explains how the feature works, where it matters and what to watch out for, then ends with a runnable example. Run it, read the output, and compare it with the notes on what to notice. Run the cells in order.

**Prerequisites:** [PySpark 2: reading, SQL and cleaning](pyspark-02-reading-sql-and-cleaning.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC) and `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-foundations.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

## Choose a join type by what happens to unmatched rows

Every join type answers one question: what happens to a row that finds no match on the other side?

| `how` | keeps | typical use |
| --- | --- | --- |
| `inner` | matched rows only | facts that must have a dimension row |
| `left` | every left row; nulls where the right side is missing | enrich a fact without losing it |
| `right` | every right row | rarely; swap the inputs and use `left` |
| `full` | every row from both sides | reconcile a source against a target |
| `left_semi` | left rows that have a match, left columns only, never duplicated | "customers who ordered" |
| `left_anti` | left rows with no match | "customers who never ordered", missing keys |
| `cross` | every combination | small calendars or salt ranges only |

**Where it matters**
- **Enrichment:** a `left` join from orders to customers keeps an order even when its customer is missing.
- **Data quality:** a `left_anti` join lists fact keys missing from a dimension.
- **Reconciliation:** a `full` join between source and warehouse classifies every key as matched, missing or extra.

**Watch out for**
- An `inner` join silently drops unmatched facts. If revenue must reconcile, use `left` and count the nulls.
- `left_semi` is not an `inner` join with fewer columns: it never multiplies left rows, even when the right side has many matches.
- A `cross` join on two real tables is almost always a bug: 1 million x 1 million rows is 10^12.

### Example

Join `customers` to `orders` with every type and compare row counts. Customers 7 and 8 have no orders, so
they appear only in the types that keep unmatched left rows.

In [ ]:
counts = {
    how: customers.join(orders, "customer_id", how).count()
    for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]
}
for how, rows in counts.items():
    print(f"{how:<10} {rows:>3} rows")

customers.join(orders, "customer_id", "left_anti").show()  # who has never ordered

enriched = orders.join(customers.select("customer_id", "name", "region"), "customer_id", "left")
print("orders:", orders.count(), "| enriched orders:", enriched.count())

**What to notice:** `inner` and `right` keep the 12 orders. `left` and `full` add customers 7 and 8 with null order columns, so
14 rows. `left_semi` returns the 6 customers who ordered, once each, although customer 1 has three orders;
`left_anti` returns the other 2. Enriching orders with a `left` join keeps all 12 because `customer_id` is
unique in `customers`.

## Handle null keys, duplicates and ambiguous columns

Join conditions use SQL equality, and `NULL = NULL` is unknown, so null keys never match. When nulls should
match each other, use the null-safe operator `eqNullSafe` (`<=>` in SQL).

How you write the condition decides the output columns. `on="customer_id"` (a name or list of names) keeps
one copy of the key. An expression such as `left["k"] == right["k"]` keeps both, and a later reference to
`k` is ambiguous. Use names when the key columns share a name, and aliases when they do not.

**Where it matters**
- **Composite keys with optional parts,** such as `(country, region)` where region can be null.
- **Self joins** (employees to managers), which always need aliases.

**Watch out for**
- A table full of null keys (guest checkouts, unknown customers) all hash to one partition and skew the join.
- `AMBIGUOUS_REFERENCE` errors after a join on an expression. Select the columns you need through their aliases.

### Example

Null keys do not match with `==` but do with `eqNullSafe`; joining by name keeps one key column, by expression two.

In [ ]:
left = spark.createDataFrame([(1, "a"), (None, "b")], "k int, left_val string")
right = spark.createDataFrame([(1, "x"), (None, "y")], "k int, right_val string")

left.join(right, "k").show()  # the null keys never match
left.join(right, left["k"].eqNullSafe(right["k"])).show()  # null-safe: they do

print("join on a name      :", left.join(right, "k").columns)
print("join on an expression:", left.join(right, left["k"] == right["k"]).columns)

l, r = left.alias("l"), right.alias("r")
tidy = l.join(r, F.col("l.k") == F.col("r.k")).select(F.col("l.k").alias("k"), "left_val", "right_val")
print("aliased and selected :", tidy.columns)

**What to notice:** The plain join returns one row: `NULL = NULL` is not true, so the two null keys never meet. `eqNullSafe`
returns both. Joining on a name keeps one `k` column; joining on an expression keeps two, and a later `k`
would be ambiguous until you select through aliases.

## Validate join cardinality before aggregating

If orders have one row per order and the customer dimension has two rows for the same key, the join produces
two rows for every matching order, and summing amounts afterwards counts revenue twice. Check that the
dimension key is unique before enriching a fact table, then compare output and input row counts.

**Where it matters**
- **Slowly changing dimensions:** a history table has several versions per key; join on the version valid at the order time, not on the key alone.
- **Vendor dimensions** that arrive with accidental duplicates.

**Watch out for**
- A total row-count check alone can miss errors in an inner join: missing matches can cancel out extra matches. Check uniqueness and unmatched keys separately.
- Fix a duplicate key with a business rule (the valid version, the latest record), not with `dropDuplicates` on an arbitrary row.

### Example

Join one order worth 2500 cents to a customer with two records. The uniqueness check finds the bad key, and
the last assertion shows what would happen to revenue if the join went ahead.

In [ ]:
orders_at_order_grain = spark.createDataFrame([(101, 7, 2500)], "order_id long, customer_id long, amount_cents long")
customer_versions = spark.createDataFrame([(7, "east"), (7, "west")], "customer_id long, region string")

customer_versions.groupBy("customer_id").count().where("count > 1").show()  # the uniqueness check

multiplied = orders_at_order_grain.join(customer_versions, "customer_id", "left")
multiplied.show()
print("revenue before the join:", orders_at_order_grain.agg(F.sum("amount_cents")).first()[0])
print("revenue after the join :", multiplied.agg(F.sum("amount_cents")).first()[0])

**What to notice:** The uniqueness check flags customer 7 before anything is joined. Joining anyway turns one order into two
rows, and summing afterwards reports 5,000 cents of revenue for a 2,500-cent order.

## Broadcast small dimensions

A **sort-merge join** shuffles both sides by the key hash, sorts each partition and merges them. A
**broadcast hash join** copies the small side to every executor, builds a hash table there, and streams the
large side past it with no shuffle at all. Spark broadcasts automatically when a side's estimated size is
below `spark.sql.autoBroadcastJoinThreshold` (10 MB by default); `F.broadcast(df)` forces it.

**Where it matters**
- **Fact-to-dimension joins:** a 3 TB orders fact and a 20 MB customer dimension. Sort-merge shuffles about 3 TB; broadcast sends 20 MB to each executor, about 2 GB in total across 100 executors.
- **Lookup tables** such as currencies, calendars and status codes.

**Watch out for**
- Broadcasting a "dimension" that is really 2 GB. Every executor and the driver must hold it, so the job fails with out-of-memory errors.
- A broadcast does not help when the large side is skewed on a key that matches many rows; see [PySpark 7](pyspark-07-monitoring-and-performance.ipynb).
- Confirm the strategy in the plan (`BroadcastHashJoin`), not from the code.

### Example

Hint the customer table and confirm that the plan broadcasts it instead of shuffling both sides.

In [ ]:
hinted = orders.join(F.broadcast(customers.select("customer_id", "region")), "customer_id", "left")
hinted.explain()

**What to notice:** The plan contains `BroadcastExchange` and `BroadcastHashJoin`, and no `SortMergeJoin`: `customers` is copied
to every executor and `orders` is never shuffled.

## Aggregate with groupBy

`groupBy(...).agg(...)` collapses rows to one per group. Spark runs a **partial aggregate** inside each map
task, shuffles the reduced rows by key, then runs the **final aggregate**. That is why `sum` and `count`
shuffle far less data than the input: each task sends one partial result per key.

Useful aggregates: `count("*")` counts rows and `count(col)` non-null values; `sum`, `avg`, `min`, `max`;
`countDistinct` and `approx_count_distinct`; `collect_list` and `collect_set` gather values into an array.
Conditional metrics use `sum(when(...))` inside the same `agg`.

**Where it matters**
- **Daily revenue and order counts** at the grain a report needs.
- **Customer metrics:** lifetime value, returned amount and last order time in one pass.

**Watch out for**
- `countDistinct` cannot be partially combined, so it ships every distinct value. Use `approx_count_distinct` when a small error is acceptable.
- `avg` ignores nulls. If missing means zero, fill it first; if it means unknown, leave it.
- `collect_list` on a high-cardinality group builds huge arrays on one executor.
- Customers with no rows do not appear in a `groupBy`. Start from the dimension with a `left` join if they must.

### Example

Summarize SKU sales across orders, and build customer metrics with a conditional sum.

In [ ]:
sku_sales = (
    order_items
    .withColumn("line_amount", F.col("qty") * F.col("unit_price"))
    .groupBy("sku")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.sum("qty").alias("units"),
        F.sum("line_amount").alias("revenue"),
        F.collect_set("order_id").alias("order_ids"),
    )
)
sku_sales.orderBy("sku").show(20, truncate=False)

customer_metrics = orders.groupBy("customer_id").agg(
    F.sum("amount").alias("lifetime_value"),
    F.sum(F.when(F.col("status") == "returned", F.col("amount")).otherwise(0)).alias("returned_amount"),
    F.max("order_ts").alias("last_order_ts"),
)
customer_metrics.orderBy("customer_id").show()

**What to notice:** `SKU-4005` appears in orders 5 and 12, so it counts two distinct orders and 110.25 of revenue;
`collect_set` lists the orders. Customer 1's lifetime value is 234.50, of which 30.00 was returned. Only 6
customers appear: 7 and 8 have no orders to group.

## Reshape with pivot, unpivot, rollup and cube

`pivot` turns the values of one column into columns: `groupBy("customer_id").pivot("status").agg(...)`.
Always pass the list of values; otherwise Spark runs an extra job to find them, and a new value silently
adds a column. `unpivot` (also called `melt`) turns columns back into rows.

`rollup("region", "status")` adds subtotals for each region and a grand total; `cube` adds every
combination. `grouping_id()` labels each output row with its level, which matters because a subtotal row
and a row whose key really is null both show a null.

**Where it matters**
- **Wide reports:** revenue per status as columns for a spreadsheet or BI tool.
- **Finance reports** with subtotals and a grand total from one query.

**Watch out for**
- A pivot without explicit values on a high-cardinality column creates thousands of columns.
- Distinguish subtotal rows from real null keys with `grouping_id()` or `grouping(col)`, never by testing for null.

### Example

Pivot revenue by status, unpivot it back, then roll revenue up by region and status. Customer 4 has no
region, so its detail rows have a null region too; `grouping_id` tells them apart from subtotals.

In [ ]:
STATUSES = ["paid", "shipped", "returned", "placed"]
by_status = orders.groupBy("customer_id").pivot("status", STATUSES).agg(F.sum("amount"))
by_status.orderBy("customer_id").show()

long_again = by_status.unpivot("customer_id", STATUSES, "status", "revenue").where("revenue is not null")
long_again.orderBy("customer_id", "status").show(5)

rolled = (
    orders.join(customers.select("customer_id", "region"), "customer_id")
    .rollup("region", "status")
    .agg(F.sum("amount").alias("revenue"), F.grouping_id().alias("level"))
)
rolled.orderBy(F.col("level"), F.col("region").asc_nulls_last(), "status").show(30)

**What to notice:** The pivot has one column per status, with null where a customer has no order of that status; `unpivot`
turns the columns back into rows. In the rollup, `level` 0 rows are detail, 1 are region subtotals (North:
549.75) and 3 is the grand total (2,659.05). Customer 4 has no region, so two detail rows show a null
region at level 0: without `grouping_id` they would look exactly like subtotals.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Which join type lists customers who have never ordered, and why not a left join?

**A.** `left_anti`. It returns customers with no match directly. A left join followed by a null filter gets the same answer but carries the right side's columns and is easier to get wrong.

**Q2.** Why do rows with null keys disappear from an inner join?

**A.** `NULL = NULL` evaluates to unknown, not true, so null keys never match. Use `eqNullSafe` when nulls should match each other.

**Q3.** A join doubled revenue. What do you check first?

**A.** Key uniqueness on the dimension side: `groupBy(key).count().where("count > 1")`. A duplicate key fans out every matching fact row.

**Q4.** When should you not broadcast the smaller table?

**A.** When it does not comfortably fit in driver and executor memory. The hint forces a copy on every executor, so a large table causes out-of-memory errors instead of a slow join.

**Q5.** Why does `sum` shuffle less data than `countDistinct`?

**A.** `sum` is combined in a partial aggregate inside each task, so only one value per key crosses the network. Distinct values cannot be combined that way and are shuffled themselves.

**Q6.** How do you tell a rollup subtotal from a row whose key is null?

**A.** Use `grouping_id()` or `grouping(col)`. Both rows show a null key, but only the subtotal has a non-zero grouping level.

## Key takeaways

- Choose the join type by what should happen to unmatched rows; use `left_anti` to find missing keys.
- Check dimension key uniqueness before joining, and compare row counts after.
- Null keys never match with `==`; use `eqNullSafe` when they should.
- Broadcast only what is genuinely small, and confirm it in the plan.
- Pass explicit values to `pivot`, and label rollup levels with `grouping_id()`.

**Next:** [PySpark 4: windows and nested data](pyspark-04-windows-and-nested-data.ipynb)